In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os
import shutil

# DIRECTORIES
BASE_DIR = '/content/drive/MyDrive/Multimodal_Stroke_Project'

# INPUT PATHS
CLEAN_METADATA_PATH = os.path.join(BASE_DIR, 'data_raw/CUBS_tech.csv')
KAGGLE_PATH = os.path.join(BASE_DIR, 'data_raw/healthcare-dataset-stroke-data.csv')
IMAGE_SOURCE_DIR = os.path.join(BASE_DIR, 'data_raw/images')

# OUTPUT PATH
OUTPUT_DIR = os.path.join(BASE_DIR, 'processed_dataset')

def generate_final_dataset():
    
    # LOAD DATASETS
    if not os.path.exists(CLEAN_METADATA_PATH):
        print(f"ERROR: Could not find {CLEAN_METADATA_PATH}")
        return
    os.makedirs(os.path.join(OUTPUT_DIR, 'images'), exist_ok=True)

    print("Loading datasets from Drive...")
    image_df = pd.read_csv(CLEAN_METADATA_PATH)
    clinical_df = pd.read_csv(KAGGLE_PATH)

    # POOLS
    sick_pool = clinical_df[clinical_df['stroke'] == 1]
    healthy_pool = clinical_df[clinical_df['stroke'] == 0]

    final_rows = []

    print(f"Pairing {len(image_df)} images...")

    # PAIRING LOOP
    for index, row in image_df.iterrows():
        filename = row['filename']
        risk_flag = row['implied_stroke_risk']

        if risk_flag == 1:
            if not sick_pool.empty:
                patient_data = sick_pool.sample(1).iloc[0].to_dict()
            else:
                continue
        else:
            patient_data = healthy_pool.sample(1).iloc[0].to_dict()

        patient_data['filename'] = filename
        patient_data['morphology_class'] = row['morphology_class']

        # MOVE IMAGES TO NEW FOLDER
        src_path = os.path.join(IMAGE_SOURCE_DIR, filename)
        dst_path = os.path.join(OUTPUT_DIR, 'images', filename)

        if os.path.exists(src_path):
            shutil.copy(src_path, dst_path)
            final_rows.append(patient_data)
        else:
            print(f"Missing image: {filename}")

    # SAVE MASTER DATASET
    if final_rows:
        master_df = pd.DataFrame(final_rows)
        save_path = os.path.join(OUTPUT_DIR, 'master_dataset.csv')
        master_df.to_csv(save_path, index=False)
        print(f"SUCCESS. Dataset created at: {save_path}")
        print(f"Total Paired Samples: {len(master_df)}")
    else:
        print("WARNING: No rows created. Check your images folder path.")

generate_final_dataset()